## **Validation plan**

В этом ноутбуке собираем валидацию из последней недели train, так как данные из нее наиболее приближены к test. Используя этот ноутбук, можно быстро проверить эффективность любой идеи на валидационной выборке, полагаясь не только на доступные в соревновании 26 процентов публичных тестов. 

Загрузка, копия из EDA

In [1]:
import polars as pl
from datetime import datetime

DATA = '../data'
train = pl.read_parquet(f'{DATA}/train.parquet')
test = pl.read_parquet(f'{DATA}/test.parquet')
ed = pl.read_parquet(f'{DATA}/event_dictionary.parquet')

def flatten(df):
    return (df.select('example_id', 'events')
              .explode('events')
              .unnest('events')
              .join(ed, on='encoded_event', how='left'))

ftr = flatten(train)
fte = flatten(test)

Отделяем последнюю неделю от train

In [2]:
ends = ftr.group_by('example_id').agg(pl.col('timestamp').max().alias('end'))

week = 7 * 24 * 60 * 60 * 1000
cut = ends['end'].max() - week

val_part = ends.filter(pl.col('end') >= cut).select('example_id')
tr  = train.join(val_part, on='example_id', how='anti') #нет в списке: наш train
val = train.join(val_part, on='example_id', how='semi') #semi оставляет строки train, у которых example_id есть в val_part, и не добавляет столбцов; здесь наш val

print(datetime.fromtimestamp(cut / 1000), len(tr), len(val))

2026-03-31 02:51:52.745000 82371 17629


## Простые правила на валидации

Для каждого примера `val` угадываем ответ без модели и считаем accuracy:
- **const** — всем самый частый ответ в `tr`;
- **view** — последний показанный продукт (событие 13), если показов нет — const;
- **click_view** — последний клик (событие 469), если кликов нет — view. (слегка умнее, чем было в EDA)

Признаки берутся из истории примера, константа — только из `tr`, ответы `val` используются только для подсчёта accuracy.

In [3]:
def last_product(f, event): #берем последнее событие нужного типа по датасету и типу события
    return (f.filter(pl.col('encoded_event') == event) 
             .group_by('example_id')
             .agg(pl.col('encoded_product').last().alias(f'last_{event}')))

top = tr['target_product'].mode()[0] #самый частый ответ на нашем трейне обрезанном без val, у нас он = 6

v = (val.join(last_product(ftr, 13), on='example_id', how='left')
        .join(last_product(ftr, 469), on='example_id', how='left')) #приклеиваем последний просмотр и последний клик

v = v.with_columns(
    const      = pl.lit(top),
    view       = pl.coalesce('last_13', pl.lit(top)), #последний показ, а если он null, то 6.
    click_view = pl.coalesce('last_469', 'last_13', pl.lit(top)), #последний клик → иначе последний показ → иначе 6.
)

for rule in ['const', 'view', 'click_view']:
    print(rule, round((v[rule] == v['target_product']).mean(), 4)) #сравниваем с ответом на val

const 0.1358
view 0.3112
click_view 0.2755


| правило | accuracy |
|---|---|
| const | 0.136 |
| view | 0.311 |
| click_view | 0.276 |

Лучшее правило — последний показ.

Клик как подсказка хуже показа: подстановка последнего клика вместо показа там, где клик есть (~44% val), снижает accuracy с 0.311 до 0.276. В EDA «последний клик» давал 0.151, потому что там не было запасного варианта: примеры без клика считались ошибкой.

`view` на последней неделе (0.311) ниже, чем на всём train в EDA (0.347).

In [4]:
print('const_55', round((v['target_product'] == 55).mean(), 4))

const_55 0.2107


Это сколько const = 55 дает на val, просто для сравнения в будущем, есть гипотеза, что на тесте при const = 55 будет ~=0

--------
*Сборка для сабмитов на тесте*

In [5]:
top_all = train['target_product'].mode()[0] #считаем топ по всему трейн - это 55

t = (test.select('example_id')
        .join(last_product(fte, 13), on='example_id', how='left')
        .join(last_product(fte, 469), on='example_id', how='left'))

t = t.with_columns(
    const      = pl.lit(top_all),
    view       = pl.coalesce('last_13', pl.lit(top_all)),
    click_view = pl.coalesce('last_469', 'last_13', pl.lit(top_all)),
)

print(top_all, len(t))

55 12359


Запишем 3 csv

In [6]:
import os

os.makedirs('../submissions', exist_ok=True)

for rule in ['const', 'view', 'click_view']:
    (t.select('example_id', pl.col(rule).alias('encoded_product'))
      .write_csv(f'../submissions/{rule}.csv'))

t.select(pl.col('const', 'view', 'click_view').null_count())

const,view,click_view
u32,u32,u32
0,0,0
